### Import packages

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

### Data and Paths

In [2]:
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"
EVAL_DIR = PROJECT_ROOT / "notebooks/Story_Quality_Checks"

sci_fi_data_path = DATA_DIR / "sci_fi_stories_cleaned.jsonl"
romance_data_path = DATA_DIR / "romance_stories_cleaned.jsonl"
literary_fiction_data_path = DATA_DIR / "lit_fiction_stories_cleaned.jsonl"

FILES = [
    sci_fi_data_path,
    romance_data_path,
    literary_fiction_data_path
]

SAMPLED_FILE = EVAL_DIR / "sampled_stories.json"
RESULTS_FILE = EVAL_DIR / "evaluation_results.json"

N_TOTAL = 100
BATCH_SIZE = 10

### Define labels

In [3]:
VALID_LABELS = {
    "1": "science fiction",
    "2": "romance",
    "3": "literary fiction"
}

### Functions

In [6]:
def load_jsonl(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def load_json(path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

### Genre prediction script

In [ ]:
# Load the sampled stories from the JSON file
sampled = load_json(SAMPLED_FILE)

results = load_json(RESULTS_FILE) or []

already_done = len(results)
remaining = N_TOTAL - already_done

print(f"\nAlready completed: {already_done}")
print(f"Remaining: {remaining}")

if remaining == 0:
    print("All 100 stories already evaluated.")
else:
    to_do_now = min(BATCH_SIZE, remaining)

    print(f"\nStarting batch of {to_do_now} stories...")

    for i in range(already_done, already_done + to_do_now):
        item = sampled[i]

        print("\n" + "="*80)
        print(f"Story {i+1}/{N_TOTAL}")
        print("="*80)
        print(item["story"])
        print("\nGuess the genre:")
        print("1 = Science Fiction")
        print("2 = Romance")
        print("3 = Literary Fiction")

        guess = None
        while guess not in VALID_LABELS:
            guess = input("Your guess (1/2/3): ").strip()

        predicted = VALID_LABELS[guess]
        true_genre = item["genre"]

        results.append({
            "predicted": predicted,
            "true": true_genre
        })

        save_json(results, RESULTS_FILE)

    print("\nBatch complete. You can safely stop now and resume later.")


if len(results) == N_TOTAL:

    correct = sum(1 for r in results if r["predicted"] == r["true"])
    accuracy = correct / len(results)

    print("\n" + "="*80)
    print("FINAL RESULTS")
    print("="*80)
    print(f"Accuracy: {accuracy:.2%} ({correct}/{len(results)})")

    confusion = defaultdict(Counter)
    for r in results:
        confusion[r["true"]][r["predicted"]] += 1

    genres = ["science fiction", "romance", "literary fiction"]

    print("\nConfusion Matrix (rows = true, columns = predicted):\n")

    print(f"{'':20}", end="")
    for g in genres:
        print(f"{g:20}", end="")
    print()

    for true in genres:
        print(f"{true:20}", end="")
        for pred in genres:
            print(f"{confusion[true][pred]:20}", end="")
        print()


Already completed: 100
Remaining: 0
All 100 stories already evaluated.

FINAL RESULTS
Accuracy: 89.00% (89/100)

Confusion Matrix (rows = true, columns = predicted):

                    science fiction     romance             literary fiction    
science fiction                       30                   0                   3
romance                                1                  31                   2
literary fiction                       1                   4                  28
